### Metahuristics Assigment


#### Part One: Hill climbing algorithm implementation

In [83]:
import numpy as np
import sys
sys.path.append("../helper")
from workload_data_loader import load_workload_instance as ld
from itertools import combinations

Helper functions for one instance
- fitness(x):evaluation of total utility
- is_feasible(x):evaluation of constraints passed over or not

In [84]:
def fitness(x:np.ndarray,p:np.ndarray):
   return p @ x

def is_feasible(x:np.ndarray,r:np.ndarray,b:np.ndarray):
    return np.all(r @ x <= b)

Helper: Generation of initial state using seed so its reproducible

In [85]:
rng = np.random.default_rng(44)
def generate_initial_state(seed,size:int,demands,capacities):
    # Will repeat until initial feasible state is acquired although the deterministic first value for seed 42 is feasible anywhay
    while True:
        x = seed.integers(0,2,size=size)
        if(is_feasible(x,demands,capacities)): 
            # print(f"fitness :{fitness(x,utilities)}")
            return x

Neighbourhood / Manipulation Functions 

In [86]:
# Neighbourhood manipulation function: 
# single bit manipulation
def generate_neighbour_single_bit(x:np.ndarray):
    neighbours = []
    for i in range(len(x)):
        neighbour = x.copy()
        neighbour[i] = 1-x[i]
        neighbours.append(neighbour)
    return np.array(neighbours)

# multi bit manipulation
# The logic is using xor bit manipulation with a mask containing 1 where you want to change values
# and 0 where you dont
# the combination function allows us to create all the combinations possible for that exact length and bit manipulation number
# 28Ck(bit manipulation number)

def generate_neighbours_multi_bit(x:np.ndarray,k:int):
    n = len(x)
    # Creating all the combinations for the length and bit manipulation number
    positions = list(combinations(range(n),k))
    # Create a mask for it 
    masks = np.zeros((len(positions),n),dtype=int)
    for i,pos in enumerate(positions):
        masks[i, list(pos)] = 1
        
    return np.bitwise_xor(x,masks)
       

Neigbourhood selction function

In [87]:
# Choosing a better neighbour
def best_neighbour(x:np.ndarray,p:np.ndarray,r:np.ndarray,b,m:callable,*args):
    neighbours = m(x,*args)
    best_state = None
    best_fitness = fitness(x,p) 
    
    for neighbour in neighbours:
        if is_feasible(neighbour,r,b):
            neighbour_fitness = fitness(neighbour,p)
            if neighbour_fitness > best_fitness:
                best_state = neighbour
                best_fitness = neighbour_fitness
                print(best_fitness)
    return best_state, best_fitness

Hill climbing algorithm

In [88]:
def hill_climbing(state,utilities,demands,capacities,mutation,*args):
    current = state.copy()
    while True:
        best_state, best_fitness = best_neighbour(current,utilities,demands,capacities,mutation,*args)
        if best_state is None:
            return current,best_fitness
        current = best_state
  

In [89]:
n, m, utilities, demands, capacities = ld(r"..\data\multi_resource_n28_m2.txt")
print("=== Dataset Ingestion Summary ===")
print(f"Number of candidate models (n) : {n}")
print(f"Number of resource bounds (m)  : {m}")
print(f"Utility scores shape (p_j)     : {utilities.shape}")
print(f"Resource demands shape (r_ij)  : {demands.shape}")
print(f"Capacity limits (b_i)          : {capacities}")


=== Dataset Ingestion Summary ===
Number of candidate models (n) : 28
Number of resource bounds (m)  : 2
Utility scores shape (p_j)     : (28,)
Resource demands shape (r_ij)  : (2, 28)
Capacity limits (b_i)          : [500. 500.]


In [90]:
x = generate_initial_state(rng,28,demands,capacities)
best_state,best_fitness = hill_climbing(x,utilities,demands,capacities,generate_neighbours_multi_bit,3)
print(best_state,best_fitness)


66263.0
76470.0
77703.0
79130.0
80680.0
81005.0
87910.0
90248.0
92908.0
94783.0
104892.0
107230.0
109890.0
111440.0
113152.0
116640.0
120730.0


128273.0
128283.0
128503.0
128663.0
128911.0
129073.0
129778.0
130103.0
130773.0
[0 0 1 0 1 0 1 1 0 1 1 0 0 1 0 0 0 0 0 0 1 0 1 1 0 0 0 0] 130773.0
